In [0]:
import pyspark.sql.functions as F
import time

print("Iniciando o processamento Distribuído...")
inicio = time.time()

caminho_base = "/Volumes/projeto/default/sd/"

# 1. Extração
print("1. Mapeando os arquivos no DBFS...")
df_orders = spark.read.csv(f"{caminho_base}orders_gigante.csv", header=True, inferSchema=True)
df_items = spark.read.csv(f"{caminho_base}items_gigante.csv", header=True, inferSchema=True)
df_customers = spark.read.csv(f"{caminho_base}customers_gigante.csv", header=True, inferSchema=True)
df_products = spark.read.csv(f"{caminho_base}olist_products_dataset.csv", header=True, inferSchema=True)
df_sellers = spark.read.csv(f"{caminho_base}olist_sellers_dataset.csv", header=True, inferSchema=True)

# 2. Transformação
print("2. Construindo o plano de JOINs e Agregações...")

# Filtrar e tratar datas
df_orders = df_orders.filter(F.col("order_status") != 'canceled')
df_orders = df_orders.withColumn("order_purchase_timestamp", F.to_timestamp(F.col("order_purchase_timestamp")))
df_orders = df_orders.withColumn("ano", F.year(F.col("order_purchase_timestamp")))
df_orders = df_orders.withColumn("mes", F.month(F.col("order_purchase_timestamp")))

# Cascata de JOINs
df_merged = df_orders.join(df_items, "order_id", "inner") \
                     .join(df_customers, "customer_id", "inner") \
                     .join(df_products, "product_id", "inner") \
                     .join(df_sellers, "seller_id", "inner")

# Agregação Complexa Multidimensional
resultado = df_merged.groupBy("customer_state", "ano", "mes", "product_category_name") \
                     .agg(
                         F.sum("price").alias("faturamento_total"),
                         F.avg("price").alias("ticket_medio"),
                         F.count("order_id").alias("quantidade_vendas")
                     ) \
                     .orderBy(F.col("faturamento_total").desc())

# 3. Carga
print("3. Acionando o cluster: Executando operações e salvando resultado...")
# Salvando o resultado de volta no DBFS para forçar o processamento completo
caminho_saida = "/Volumes/projeto/default/sd/resultado_distribuido_spark"
resultado.write.mode("overwrite").csv(caminho_saida, header=True)

fim = time.time()
tempo_total = fim - inicio

print("\nProcessamento concluído!")
print(f"Tempo total de execução (Distribuído): {tempo_total:.2f} segundos")

print("\nTop 5 Categorias/Estados com maior faturamento:")
resultado.show(5, truncate=False)

In [0]:
%pip install psutil

import psutil
import time
import threading
import pyspark.sql.functions as F

# Variáveis globais para armazenar as métricas
monitorando = True
historico_cpu = []
historico_ram = []

# Função que vai rodar em segundo plano monitorando a máquina
def monitorar_hardware():
    global historico_cpu, historico_ram, monitorando
    while monitorando:
        # Pega a % de uso da CPU e da RAM do container do Databricks
        historico_cpu.append(psutil.cpu_percent(interval=None))
        historico_ram.append(psutil.virtual_memory().percent)
        time.sleep(1) # Lê a cada 1 segundo

# 1. Inicia o monitoramento em uma Thread separada
print("Iniciando monitor de hardware...")
thread_monitor = threading.Thread(target=monitorar_hardware)
thread_monitor.start()

# ====================================================================
# SEU CÓDIGO PYSPARK AQUI DENTRO (O TESTE DE ESTRESSE)
# ====================================================================
print("Iniciando processamento Spark...")
inicio = time.time()

caminho_base = "/Volumes/projeto/default/sd/"

# Extração
df_orders = spark.read.csv(f"{caminho_base}orders_gigante.csv", header=True, inferSchema=True)
df_items = spark.read.csv(f"{caminho_base}items_gigante.csv", header=True, inferSchema=True)
df_customers = spark.read.csv(f"{caminho_base}customers_gigante.csv", header=True, inferSchema=True)
df_products = spark.read.csv(f"{caminho_base}olist_products_dataset.csv", header=True, inferSchema=True)
df_sellers = spark.read.csv(f"{caminho_base}olist_sellers_dataset.csv", header=True, inferSchema=True)

# Transformação
df_orders = df_orders.filter(F.col("order_status") != 'canceled')
df_orders = df_orders.withColumn("order_purchase_timestamp", F.to_timestamp(F.col("order_purchase_timestamp")))
df_orders = df_orders.withColumn("ano", F.year(F.col("order_purchase_timestamp")))
df_orders = df_orders.withColumn("mes", F.month(F.col("order_purchase_timestamp")))

df_merged = df_orders.join(df_items, "order_id", "inner") \
                     .join(df_customers, "customer_id", "inner") \
                     .join(df_products, "product_id", "inner") \
                     .join(df_sellers, "seller_id", "inner")

resultado = df_merged.groupBy("customer_state", "ano", "mes", "product_category_name") \
                     .agg(
                         F.sum("price").alias("faturamento_total"),
                         F.avg("price").alias("ticket_medio"),
                         F.count("order_id").alias("quantidade_vendas")
                     ) \
                     .orderBy(F.col("faturamento_total").desc())

# Carga (Onde o processador frita)
caminho_saida = "/Volumes/projeto/default/sd/resultado_distribuido_spark"
resultado.write.mode("overwrite").csv(caminho_saida, header=True)

tempo_total = time.time() - inicio
# ====================================================================

# 2. Para o monitoramento
monitorando = False
thread_monitor.join()

# 3. Printa os Resultados do Hardware
print("\n" + "="*50)
print(" RELATÓRIO DE CONSUMO DE HARDWARE (DATABRICKS)")
print("="*50)
print(f"Tempo Total de Execução : {tempo_total:.2f} segundos")
print(f"Pico de uso da CPU      : {max(historico_cpu):.2f}%")
print(f"Uso médio da CPU        : {sum(historico_cpu)/len(historico_cpu):.2f}%")
print(f"Pico de uso da Memória  : {max(historico_ram):.2f}%")
print(f"Uso médio da Memória    : {sum(historico_ram)/len(historico_ram):.2f}%")
print("="*50)